In [ ]:
import pyorbital
from pyorbital.orbital import Orbital
import datetime as dt
from matplotlib import colormaps as cmap
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
import numpy as np
import pandas as pd
import bisect
import uuid
from enum import Enum
from shapely.geometry import Polygon
from shapely.plotting import patch_from_polygon

from fame import *
import copy

In [ ]:
# Behind the scenes, this pulls from Celestrak if we do not specify tle_file

satellites = [
    Satellite("LOFT YAM-3", Orbital("YAM-3", tle_file="TLEs.txt")),
    Satellite("LOFT YAM-5", Orbital("YAM-5", tle_file="TLEs.txt")),
    Satellite("LOFT YAM-6", Orbital("YAM-6", tle_file="TLEs.txt")),
    Satellite("LOFT YAM-7", Orbital("YAM-7", tle_file="TLEs.txt")),
    Satellite("LOFT YAM-8", Orbital("YAM-8", tle_file="TLEs.txt")),
    Satellite("LOFT YAM-10", Orbital("YAM-10", tle_file="TLEs.txt")),
    Satellite("Ubotica CogniSat-6 HAMMER", Orbital("HAMMER", tle_file="TLEs.txt")),
    # "LOFT YAM-3": Orbital("YAM-3"),
]

In [ ]:

ground_stations = [
    Location(
        -79.55,
        8.9833,
        0.028,
        "KSAT Panama",
    ),
    Location(
        -51.73363,
        64.182789,
        0,
        "KSAT Nuuk",
    ),
    Location(
        2.53219,
        -72.01243,
        0,
        "KSAT Troll",
    ),
    Location(
        142.3689,
        43.8,
        0,
        "KSAT Hokkaido",
    ),
    Location(
        103.9915,
        1.3661,
        0,
        "KSAT Singapore",
    ),
    Location(
        -70.85021,
        -52.93279,
        0,
        "KSAT Punta Arenas",
    ),
    Location(
        127.7766,
        26.4055,
        0,
        "KSAT Okinawa",
    ),
    Location(
        57.5565,
        -20.1142,
        0,
        "KSAT Mauritius",
    ),
    Location(
        22.62216,
        37.84604,
        0,
        "KSAT Nemea",
    ),
    Location(
        31.12509,
        70.36779,
        0,
        "KSAT Vardo",
    ),
    Location(
        15.39964,
        78.22875,
        0,
        "KSAT Svalbard",
    ),
]

# ground_station_opportunities = [
#     observation_request(
#         lon_deg=gs.lon_deg,
#         lat_deg=gs.lat_deg,
#         alt_km=gs.alt_km,
#         min_time=dt.datetime.now(dt.timezone.utc),
#         max_time=dt.datetime.now(dt.timezone.utc)+ dt.timedelta(seconds=3600*24*2)
#     )
#     for gs in ground_stations
# ]

In [ ]:
stride_s = 60
plot_range_s = 10800

# min_time = dt.datetime.now(dt.timezone.utc).replace(tzinfo=None)

min_time = dt.datetime(2026, 1, 14, 16, 55, 36, 841169)
max_time = min_time + dt.timedelta(hours=24)

In [ ]:


cities_of_the_world = pd.read_csv("simplemaps_worldcities_basicv1.901/worldcities.csv")
sampled_world_cities = cities_of_the_world.sample(n=100,weights='population',axis=0, random_state=0)
one_hundred_sampled_cities = [
    ObservationRequest(city[1].lng, city[1].lat, min_time=min_time, max_time=max_time, alt_km=0.307, request_name=city[1].city,)
    for city in sampled_world_cities.iterrows()
]

# Simulation

Let's talk about simulation.

We want an event-based sim. There is a global ordered timeline and the sim jumps from event to event.

Continuous transitions (power, thermal, etc) are discretized in this setting.

We need a few entities here.

Agent: a satellite, a constellation manager, a requestor. 

Event: something on the ground turning on or off.

Communication: something that affects the state of two entities.

Observation: a query of an agent to (event, empty set). If we want to be fancy, a query of an agent to a region, where events are or are 
not associated with regions.

Prototype:
- A Satellite class with a list of Observations
- A ConstellationManager class with a list of Satellites which can query Observations and add new ones.
  - A ConstellationManager class that updates the Satellites' Observations when a Communication event occurs.
- An Observation of a Region.
    - Observe. Returns an image of the region.
    - Search. Returns True if there is an Event in the Region. The event is stored.
    - Monitor. Returns True if the event is still active.
    - Something for moving events?
- Phenomenon: something with a spatial position, start time, end time, potentially internal states.

The simulator holds:
- Agents (Satellites, ConstellationManagers)
- Phenomena.
- A list of Events (Observation, Communication, Phenomenon transitions), encoded as functions.
The output of the functions affects the agents and phenomena.

We need to define:

- An Observation Opportunity (observation_opportunity)
- An Observer, which has an Orbit, a list of Observation Opportunities to execute, a list of ObservedEvents it has observed (with their state), and a list of DataProducts.
- A ConstellationManager, which has a list of ObserverStates (Orbit, ObservationOpportunities) and a list of CommunicationOpportunities with Observers. When there is a CommunicationOpportunity the ConstellationManager can push an updated list of ObservationOpportunities.
- An Event, with a lonlatalt, a State (enum), and times for StateTransitions.
- 

Something simple. 
- [X] Create a world.
- [X] Add satellites to it.
- [X] Add observation opportunities to the satellites manually
- [X] Click until out of events
- [X] Add a better way to add observation opportunities!
- [X] Add a ConstellationScheduler 

In [ ]:
phenomena = [
    Phenomenon(
        lon_deg = -118.,
        lat_deg = 34.,
        alt_km=0.307,
        start_time=min_time,
        end_time=max_time,

    ),
    Phenomenon(8., 45.,  alt_km=0.216, start_time=min_time, end_time=max_time),
    Phenomenon(-80., 34., alt_km=0.041, start_time=min_time, end_time=max_time),
]

In [ ]:
phenomena_cities = [
    Phenomenon(
        lon_deg = city.lon_deg,
        lat_deg = city.lat_deg,
        alt_km=city.alt_km,
        start_time=city.min_time,
        end_time=city.max_time,
        name=city.name
    ) for city in one_hundred_sampled_cities
]

In [ ]:
one_hundred_sampled_cities[0].min_time

datetime.datetime(2026, 1, 14, 16, 55, 36, 841169)

In [ ]:
satellite_agents_simple = [
    Satellite("LOFT YAM-3", Orbital("YAM-3", tle_file="TLEs.txt")),
    Satellite("LOFT YAM-5", Orbital("YAM-5", tle_file="TLEs.txt")),
    Satellite("LOFT YAM-6", Orbital("YAM-6", tle_file="TLEs.txt")),
    Satellite("LOFT YAM-7", Orbital("YAM-7", tle_file="TLEs.txt")),
    Satellite("LOFT YAM-8", Orbital("YAM-8", tle_file="TLEs.txt")),
    Satellite("LOFT YAM-10", Orbital("YAM-10", tle_file="TLEs.txt")),
    Satellite("Ubotica CogniSat-6 HAMMER", Orbital("HAMMER", tle_file="TLEs.txt")),
    # "LOFT YAM-3": Orbital("YAM-3"),
]

In [ ]:
world = World(satellites = satellite_agents_simple, phenomena=phenomena)

Manually add observation opportunities

In [ ]:
obs_requests = [
    ObservationRequest(-118., 34., min_time=min_time, max_time=max_time, alt_km=0.307),
    ObservationRequest(8., 45., min_time=min_time, max_time=max_time, alt_km=0.216),
    ObservationRequest(-80., 34., min_time=min_time, max_time=max_time, alt_km=0.041),
]

In [ ]:
observation_opportunities = find_observation_opportunities(obs_requests, satellites)

In [ ]:
best_request = {r: None for r in obs_requests}

for request, passes in observation_opportunities.items():
    _best_quality = - np.inf
    _best_satellite = None
    _best_pass = None
    for satellite, satpasses in passes.items():
        for satpass in satpasses:
            _quality = observation_quality(satpass.highest)
            if _quality >= _best_quality:
                _best_quality = _quality
                _best_satellite = satellite
                _best_pass = satpass
            # print("{}: quality {}".format(satpass.highest, observation_quality(satpass.highest)))
    best_request[request] = (_best_satellite, _best_pass)

In [ ]:
for req, opp in best_request.items():
    for _sat in world.satellites:
        if _sat.name == opp[0].name:
            _opportunity = opp[1].highest
            print("{} {} {}".format(_sat.name, req, _opportunity))

            schedule_observation(world, _opportunity)

LOFT YAM-8 Request  Observation  at 2026-01-15 07:23:40.590938 by LOFT YAM-8 with InstrumentType.RGB. Look angle 286.7715401333793 | 37.97139460052666 az/dec deg, zenith angle 164.65043755426706 deg, range 821.1528572600462 km, duration 0:01:00
Ubotica CogniSat-6 HAMMER Request  Observation  at 2026-01-15 02:09:23.939248 by Ubotica CogniSat-6 HAMMER with InstrumentType.RGB. Look angle 271.6996286032928 | 49.585105266385426 az/dec deg, zenith angle 140.7625206274004 deg, range 389.60162997048013 km, duration 0:01:00
Ubotica CogniSat-6 HAMMER Request  Observation  at 2026-01-15 08:08:41.141603 by Ubotica CogniSat-6 HAMMER with InstrumentType.RGB. Look angle 251.11656608476758 | 51.532434220826566 az/dec deg, zenith angle 142.68100312718087 deg, range 374.81621922843146 km, duration 0:01:00


In [ ]:
retcode = 1
while (retcode !=0):
    print("\nTick!")
    print(world.events)
    # print({sat.name: sat.scheduled_observations for sat in world.agents if len(sat.scheduled_observations)})
    retcode = world.tick()


Tick!
[Event Obs, sat Ubotica CogniSat-6 HAMMER at 2026-01-15 02:09:23.939248, Event Obs, sat LOFT YAM-8 at 2026-01-15 07:23:40.590938, Event Obs, sat Ubotica CogniSat-6 HAMMER at 2026-01-15 08:08:41.141603]
Executing Event Obs, sat Ubotica CogniSat-6 HAMMER at 2026-01-15 02:09:23.939248

Tick!
[Event Unlock satellite after obs, sat Ubotica CogniSat-6 HAMMER at 2026-01-15 02:10:23.939248, Event Obs, sat LOFT YAM-8 at 2026-01-15 07:23:40.590938, Event Obs, sat Ubotica CogniSat-6 HAMMER at 2026-01-15 08:08:41.141603]
Executing Event Unlock satellite after obs, sat Ubotica CogniSat-6 HAMMER at 2026-01-15 02:10:23.939248

Tick!
[Event Obs, sat LOFT YAM-8 at 2026-01-15 07:23:40.590938, Event Obs, sat Ubotica CogniSat-6 HAMMER at 2026-01-15 08:08:41.141603]
Executing Event Obs, sat LOFT YAM-8 at 2026-01-15 07:23:40.590938

Tick!
[Event Unlock satellite after obs, sat LOFT YAM-8 at 2026-01-15 07:24:40.590938, Event Obs, sat Ubotica CogniSat-6 HAMMER at 2026-01-15 08:08:41.141603]
Executing E

In [ ]:
[sat.known_phenomena for sat in world.satellites]

[[], [], [], [], [], [], []]

In [ ]:
satellite_agents_world = [
    Satellite("LOFT YAM-3", Orbital("YAM-3", tle_file="TLEs.txt")),
    Satellite("LOFT YAM-5", Orbital("YAM-5", tle_file="TLEs.txt")),
    Satellite("LOFT YAM-6", Orbital("YAM-6", tle_file="TLEs.txt")),
    Satellite("LOFT YAM-7", Orbital("YAM-7", tle_file="TLEs.txt")),
    Satellite("LOFT YAM-8", Orbital("YAM-8", tle_file="TLEs.txt")),
    Satellite("LOFT YAM-10", Orbital("YAM-10", tle_file="TLEs.txt")),
    Satellite("Ubotica CogniSat-6 HAMMER", Orbital("HAMMER", tle_file="TLEs.txt")),
    # "LOFT YAM-3": Orbital("YAM-3"),
]

In [ ]:
world_with_cities = World(satellites = satellite_agents_world, phenomena=phenomena_cities)



In [ ]:
best_request_cities = {r: None for r in one_hundred_sampled_cities}

observation_opportunities_cities = find_observation_opportunities(one_hundred_sampled_cities, satellite_agents_world)

for request, passes in observation_opportunities_cities.items():
    _best_quality = - np.inf
    _best_satellite = None
    _best_pass = None
    for satellite, satpasses in passes.items():
        for satpass in satpasses:
            _quality = observation_quality(satpass.highest)
            if _quality >= _best_quality:
                _best_quality = _quality
                _best_satellite = satellite
                _best_pass = satpass
            # print("{}: quality {}".format(satpass.highest, observation_quality(satpass.highest)))
    best_request_cities[request] = (_best_satellite, _best_pass)

In [ ]:
for req, opp in best_request_cities.items():
    for _sat in world_with_cities.satellites:
        # Give the event to the right agent
        if _sat.name == opp[0].name:
            # The opportunity is the middle of the pass
            _opportunity = opp[1].highest
            # print("{} {} {}".format(_sat.name, req, _opportunity))

            # This is quite redundant. What you want is to maintain events for individual agents and then a global copy, right?
            _sat.scheduled_observations.append(_opportunity)
            _event = Event(
                name="Observation for {}: {}".format(opp[0].name, opp[1]),
                time = _opportunity.time,
                # The magic is here: we add an event that does an observation and stores the result in the agent's known_phenomena bin
                # Note the kludge of default inputs to make sure the closure works and we capture the variables at the time of creation
                # action_callable = lambda _opp=_opportunity, _satname=_sat.name: print("{} with {}".format(_opp, _satname)) #_sat.known_phenomena.append(world.do_observation(_opportunity, _sat))
                action_callable = lambda _opp=_opportunity: world_with_cities.do_observation(_opp)
            )
            world_with_cities.add_event(_event)

In [ ]:
retcode = 1
while (retcode !=0):
    # print("Tick!")
    # print(world_cities.events)
    # print({sat.name: sat.scheduled_observations for sat in world.agents if len(sat.scheduled_observations)})
    retcode = world_with_cities.tick()

Executing Event Observation for Ubotica CogniSat-6 HAMMER: Pass start: 2026-01-14 16:59:31.660167, highest: 2026-01-14 17:03:40.372295, fall: 2026-01-14 17:07:57.541630 at 2026-01-14 17:03:40.372295
Executing Event Observation for Ubotica CogniSat-6 HAMMER: Pass start: 2026-01-14 16:59:46.297132, highest: 2026-01-14 17:03:41.733067, fall: 2026-01-14 17:07:41.743018 at 2026-01-14 17:03:41.733067
Executing Event Observation for Ubotica CogniSat-6 HAMMER: Pass start: 2026-01-14 16:59:50.714941, highest: 2026-01-14 17:03:42.965321, fall: 2026-01-14 17:07:39.783537 at 2026-01-14 17:03:42.965321
Executing Event Observation for LOFT YAM-7: Pass start: 2026-01-14 17:03:54.043784, highest: 2026-01-14 17:09:26.895906, fall: 2026-01-14 17:14:52.174746 at 2026-01-14 17:09:26.895906
Executing Event Observation for LOFT YAM-7: Pass start: 2026-01-14 17:04:06.514154, highest: 2026-01-14 17:09:37.237135, fall: 2026-01-14 17:15:08.310277 at 2026-01-14 17:09:37.237135
Executing Event Observation for LOF

In [ ]:
satellite_agents_sched = [
    Satellite("LOFT YAM-3", Orbital("YAM-3", tle_file="TLEs.txt")),
    Satellite("LOFT YAM-5", Orbital("YAM-5", tle_file="TLEs.txt")),
    Satellite("LOFT YAM-6", Orbital("YAM-6", tle_file="TLEs.txt")),
    Satellite("LOFT YAM-7", Orbital("YAM-7", tle_file="TLEs.txt")),
    Satellite("LOFT YAM-8", Orbital("YAM-8", tle_file="TLEs.txt")),
    Satellite("LOFT YAM-10", Orbital("YAM-10", tle_file="TLEs.txt")),
    Satellite("Ubotica CogniSat-6 HAMMER", Orbital("HAMMER", tle_file="TLEs.txt")),
    # "LOFT YAM-3": Orbital("YAM-3"),
]

world_with_scheduler = World(satellites = satellite_agents_sched, phenomena=phenomena)

scheduler = ConstellationGroundScheduler(satellites=satellite_agents_sched, ground_stations=ground_stations, world=world_with_scheduler)

world_with_scheduler.add_constellation(scheduler)

In [ ]:
for request in obs_requests:
    scheduler.schedule_request(
        request=request,
        current_time=min_time,
        callback_request_scheduled=lambda obs: print("Request {} scheduled for observation {}!".format(request, obs)),
        callback_request_ready=lambda dp: print("Request {} ready with DP {}!".format(request, dp)),
        )

[Constellation] Scheduling request Request 
 [Constellation] Best request: Pass start: 2026-01-15 07:17:54.723674, highest: 2026-01-15 07:23:40.590938, fall: 2026-01-15 07:29:26.542330 with LOFT YAM-8
Request Request  scheduled for observation Observation  at 2026-01-15 07:23:40.590938 by LOFT YAM-8 with InstrumentType.RGB. Look angle 286.7715401333793 | 37.97139460052666 az/dec deg, zenith angle 164.65043755426706 deg, range 821.1528572600462 km, duration 0:01:00!
[Constellation] Scheduling request Request 
 [Constellation] Best request: Pass start: 2026-01-15 02:05:02.757487, highest: 2026-01-15 02:09:23.939248, fall: 2026-01-15 02:13:35.867205 with Ubotica CogniSat-6 HAMMER
Request Request  scheduled for observation Observation  at 2026-01-15 02:09:23.939248 by Ubotica CogniSat-6 HAMMER with InstrumentType.RGB. Look angle 271.6996286032928 | 49.585105266385426 az/dec deg, zenith angle 140.7625206274004 deg, range 389.60162997048013 km, duration 0:01:00!
[Constellation] Scheduling re

In [ ]:
scheduler.schedule_downlinks(current_time=min_time, max_time=max_time)

In [ ]:
retcode = 1
while (retcode !=0):
    # print("Tick!")
    retcode = world_with_scheduler.tick(print_forbidden_prefixes=["Downlink", "End of downlink"])
    # print(world_with_scheduler.events)

Executing Event Uplink, station KSAT Hokkaido to sat Ubotica CogniSat-6 HAMMER at 2026-01-14 17:05:35.695583
Executing Event Uplink, station KSAT Hokkaido to sat Ubotica CogniSat-6 HAMMER at 2026-01-14 17:05:35.695583
Executing Event Unlock uplink, station KSAT Hokkaido to sat Ubotica CogniSat-6 HAMMER at 2026-01-14 17:07:28.990894
Executing Event Unlock uplink, station KSAT Hokkaido to sat Ubotica CogniSat-6 HAMMER at 2026-01-14 17:07:28.990894
Executing Event Uplink, station KSAT Troll to sat LOFT YAM-8 at 2026-01-14 17:38:03.476271
Executing Event Unlock uplink, station KSAT Troll to sat LOFT YAM-8 at 2026-01-14 17:40:05.408832
Executing Event Obs, sat Ubotica CogniSat-6 HAMMER at 2026-01-15 02:09:23.939248
Executing Event Unlock satellite after obs, sat Ubotica CogniSat-6 HAMMER at 2026-01-15 02:10:23.939248
Spacecraft Ubotica CogniSat-6 HAMMER has 1 data products to download
  Downlinked Observation  at 2026-01-15 02:09:23.939248 by Ubotica CogniSat-6 HAMMER with InstrumentType.RG

In [ ]:
# scheduler.requests

In [ ]:
scheduler._requests

,request,satellite,observation,uplink,downlink,status,data_product,scheduled_callback,unscheduled_callback,ready_callback
0,Request,LOFT YAM-8,Observation at 2026-01-15 07:23:40.590938 by ...,"Pass start: 2026-01-14 17:36:02.426338, highes...","Pass start: 2026-01-15 08:28:09.433927, highes...",ObservationStatus.DATA_RECEIVED,"[: Lon -118.0°, lat 34.0°, alt 0.307 km, hdg N...",<function <lambda> at 0x000001A81471EFC0>,<function ConstellationGroundScheduler.<lambda...,<function <lambda> at 0x000001A81471F060>
1,Request,Ubotica CogniSat-6 HAMMER,Observation at 2026-01-15 02:09:23.939248 by ...,"Pass start: 2026-01-14 17:03:38.908884, highes...","Pass start: 2026-01-15 03:09:00.545542, highes...",ObservationStatus.DATA_RECEIVED,"[: Lon 8.0°, lat 45.0°, alt 0.216 km, hdg None...",<function <lambda> at 0x000001A81471F100>,<function ConstellationGroundScheduler.<lambda...,<function <lambda> at 0x000001A81471F1A0>
2,Request,Ubotica CogniSat-6 HAMMER,Observation at 2026-01-15 08:08:41.141603 by ...,"Pass start: 2026-01-14 17:03:38.908884, highes...","Pass start: 2026-01-15 09:16:10.626266, highes...",ObservationStatus.DATA_RECEIVED,"[: Lon -80.0°, lat 34.0°, alt 0.041 km, hdg No...",<function <lambda> at 0x000001A8148D3880>,<function ConstellationGroundScheduler.<lambda...,<function <lambda> at 0x000001A8148D3920>


Where do we go from here?

The obvious: add conflicts between observations. For now, we can stay in discrete land where an opportunity is an instantaneous thing. Or we can create multiple copies all along the path.
Conflict: two opportunities are on the same satellite and closer than some amount of time. The time should account for (i) some fixed setup (think "wait for the mirror to stop flapping") plus some time related to reorientation.
This misses the fact that one could point the spacecraft slightly away from the target and still get it. 
Can we get that in pre-processing?

Solve the one-off scheduling problem with constraints (greedy, find the best observation that is feasible).

Solve the one-off scheduling problem with constraints including comms (greedy, find the best observation that is feasible after we can talk to a given satellite).

Solve the batch scheduling problem with constraints (ILP? For old times' sake).

Multiple instruments. Spacecraft should have an instrument attached. Requests and S/C have an instrument.

Follow-on requests. A detection causes a follow-up request. Simulate.

Write up the three cases of interest:
- Submit a request and you immediately hear back. Unsubmitting requests is free.
    - Use your favorite black-box scheduling algorithm. Submitting a request==evaluating a constraint.
- Submit a request and you immediately hear back. Unsubmitting requests is expensive.
    - Use your favorite non-backtracking scheduling algorithm. Once you choose, no regrets.
- Submit a request and you don't hear back. This is a DMU problem.
    - State:
    - Actions: schedule an observation on a satellite.
    - Transitions: from "unscheduled" to "scheduled" to "executed" or "rejected" for every observation. 
    - Observations: when a file is downloaded, we find out.
    - Rewards: k if we get an observation, 0 otherwise.

- [ ] Constellation: 
  - Input:
      - [X] A new request
      - [X] A schedule of observations assigned to agents
      - [X] (can compute) Alternate windows for the assigned observations
  - Output:
      - A new schedule of observations assigned to agents
      - [X] A bool indicating whether the request was assigned
  - Formulate the optimization problem:
      - Identify which observations can be unscheduled (are not yet committed)
      - Formulate the profit-maximizing problem of assigning everything
      - Solve the problem
      - Check if the new observation is in.
- [ ] Broker
  - Input:
      - A new request
  - Output:
      - Time when the request is scheduled
  - List all windows across all constellations
  - Pick the best one
  - Send the request
- API:
    - [ ] Constellation
        - Input:
            - [X] Submit a request
            - [X] Retrieve a request status
        - Output:
            - [X] Report data acquired
            - [X] Report event
            - [X] Report unscheduled task
    - [ ] Broker
        - Input:
            - Submit a workflow request
            - Status update on observation request
        - Output
            -  Observation requests
            -  Status updates on workflow requests

In [ ]:
# Let's have multiple constellations!

# Behind the scenes, this pulls from Celestrak if we do not specify tle_file

satellites_LOFT = [
    # Satellite("LOFT YAM-3", Orbital("YAM-3", tle_file="TLEs.txt")),
    # Satellite("LOFT YAM-5", Orbital("YAM-5", tle_file="TLEs.txt")),
    Satellite("LOFT YAM-6", Orbital("YAM-6", tle_file="TLEs.txt")),
    # Satellite("LOFT YAM-7", Orbital("YAM-7", tle_file="TLEs.txt")),
    # Satellite("LOFT YAM-8", Orbital("YAM-8", tle_file="TLEs.txt")),
    Satellite("LOFT YAM-10", Orbital("YAM-10", tle_file="TLEs.txt")),
    # Satellite("Ubotica CogniSat-6 HAMMER", Orbital("HAMMER", tle_file="TLEs.txt")),
    # "LOFT YAM-3": Orbital("YAM-3"),
]

satellites_ubotica = [
    Satellite("Ubotica CogniSat-6 HAMMER", Orbital("HAMMER", tle_file="TLEs.txt")),
    ]

satellites_aerospace = [
        Satellite("AEROCUBE 18A", Orbital("AEROCUBE 18A", tle_file="TLEs.txt")),
        Satellite("AEROCUBE 18B", Orbital("AEROCUBE 18B", tle_file="TLEs.txt")),   

]

satellites_capella = [
        Satellite("AEROCUBE 18A", Orbital("AEROCUBE 18A", tle_file="TLEs.txt")),
        Satellite("AEROCUBE 18B", Orbital("AEROCUBE 18B", tle_file="TLEs.txt")),   

]

satellites_multischedulers = satellites_LOFT+satellites_ubotica+satellites_aerospace

In [ ]:
world_with_schedulers = World(satellites = satellites_multischedulers, phenomena=phenomena_cities)

scheduler_LOFT = ConstellationGroundScheduler(satellites=satellites_LOFT, ground_stations=ground_stations, world=world_with_schedulers, name="LOFT")
scheduler_ubotica = ConstellationGroundScheduler(satellites=satellites_ubotica, ground_stations=ground_stations, world=world_with_schedulers, name="UBOTICA")
scheduler_aerospace = ConstellationGroundScheduler(satellites=satellites_aerospace, ground_stations=ground_stations, world=world_with_schedulers, name="AC")

world_with_schedulers.add_constellation(scheduler_LOFT)
world_with_schedulers.add_constellation(scheduler_ubotica)
world_with_schedulers.add_constellation(scheduler_aerospace)

In [ ]:
sampled_world_cities_LOFT = cities_of_the_world.sample(n=50,weights='population',axis=0, random_state=0)
sampled_world_cities_ubotica = cities_of_the_world.sample(n=50,weights='population',axis=0, random_state=1)
sampled_world_cities_aerospace = cities_of_the_world.sample(n=50,weights='population',axis=0, random_state=2)

obs_request_LOFT = [
    ObservationRequest(city[1].lng, city[1].lat, min_time=min_time, max_time=max_time, alt_km=0.307, request_name=city[1].city,)
    for city in sampled_world_cities_LOFT.iterrows()
]

obs_request_ubotica = [
    ObservationRequest(city[1].lng, city[1].lat, min_time=min_time, max_time=max_time, alt_km=0.307, request_name=city[1].city,)
    for city in sampled_world_cities_ubotica.iterrows()
]

obs_request_aerospace = [
    ObservationRequest(city[1].lng, city[1].lat, min_time=min_time, max_time=max_time, alt_km=0.307, request_name=city[1].city,)
    for city in sampled_world_cities_aerospace.iterrows()
]

In [ ]:
# sim_start_time = dt.datetime.now(dt.timezone.utc).replace(tzinfo=None)
sim_start_time = dt.datetime(2026, 1, 14, 20, 22, 55, 552917)

for request in obs_request_LOFT:
    scheduler_LOFT.schedule_request(
        request=request,
        current_time=sim_start_time,
        callback_request_scheduled=lambda obs: print("[LOFT] Request {} scheduled for observation {}!".format(request, obs)),
        callback_request_ready=lambda dp: print("[LOFT] Request {} ready with DP {}!".format(request, dp)),
        )
    
for request in obs_request_ubotica:
    scheduler_ubotica.schedule_request(
        request=request,
        current_time=sim_start_time,
        callback_request_scheduled=lambda obs: print("[UBOTICA] Request {} scheduled for observation {}!".format(request, obs)),
        callback_request_ready=lambda dp: print("[UBOTICA] Request {} ready with DP {}!".format(request, dp)),
        )
    
for request in obs_request_aerospace:
    scheduler_aerospace.schedule_request(
        request=request,
        current_time=sim_start_time,
        callback_request_scheduled=lambda obs: print("[AC] Request {} scheduled for observation {}!".format(request, obs)),
        callback_request_ready=lambda dp: print("[AC] Request {} ready with DP {}!".format(request, dp)),
        )

# for request in obs_request_LOFT:
#     scheduler_LOFT.schedule_request(
#         request=request,
#         current_time=sim_start_time,
#         callback_request_scheduled=lambda obs: print("[LOFT] Request {} scheduled for observation {}!".format(request, obs)),
#         callback_request_ready=lambda dp: print("[LOFT] Request {} ready with DP {}!".format(request, dp)),
#         )
    
# for request in obs_request_ubotica:
#     scheduler_ubotica.schedule_request(
#         request=request,
#         current_time=sim_start_time,
#         callback_request_scheduled=lambda obs: print("[UBOTICA] Request {} scheduled for observation {}!".format(request, obs)),
#         callback_request_ready=lambda dp: print("[UBOTICA] Request {} ready with DP {}!".format(request, dp)),
#         )
    
# for request in obs_request_aerospace:
#     scheduler_aerospace.schedule_request(
#         request=request,
#         current_time=sim_start_time,
#         callback_request_scheduled=lambda obs: print("[AC] Request {} scheduled for observation {}!".format(request, obs)),
#         callback_request_ready=lambda dp: print("[AC] Request {} ready with DP {}!".format(request, dp)),
#         )

[LOFT] Scheduling request Request Korla
   No contacts for this satellite! Maybe we were too greedy
   No contacts for this satellite! Maybe we were too greedy
   No contacts for this satellite! Maybe we were too greedy
   No contacts for this satellite! Maybe we were too greedy
 [LOFT] Best request: Pass start: 2026-01-15 08:33:44.322118, highest: 2026-01-15 08:38:58.486645, fall: 2026-01-15 08:44:14.088448 with LOFT YAM-6
[LOFT] Request Request Korla scheduled for observation Observation  at 2026-01-15 08:38:58.486645 by LOFT YAM-6 with InstrumentType.RGB. Look angle 96.10422088414224 | 28.379991664374835 az/dec deg, zenith angle 70.08175492178913 deg, range 883.4643581158363 km, duration 0:01:00!
[LOFT] Scheduling request Request Đà Lạt
   No contacts for this satellite! Maybe we were too greedy
   No contacts for this satellite! Maybe we were too greedy
   No contacts for this satellite! Maybe we were too greedy
 [LOFT] Best request: Pass start: 2026-01-15 07:07:24.489601, highest:

In [ ]:
# Downlinks are now built-in to observations!

# scheduler_LOFT.schedule_downlinks(current_time=sim_start_time, max_time=sim_start_time+dt.timedelta(hours=36))
# scheduler_ubotica.schedule_downlinks(current_time=sim_start_time, max_time=sim_start_time+dt.timedelta(hours=36))
# scheduler_aerospace.schedule_downlinks(current_time=sim_start_time, max_time=sim_start_time+dt.timedelta(hours=36))

In [ ]:
world_with_schedulers.events

[Event Uplink, station KSAT Troll to sat AEROCUBE 18B at 2026-01-14 20:29:48.015687,
 Event Uplink, station KSAT Troll to sat AEROCUBE 18B at 2026-01-14 20:29:48.015687,
 Event Uplink, station KSAT Troll to sat AEROCUBE 18B at 2026-01-14 20:29:48.015687,
 Event Uplink, station KSAT Troll to sat AEROCUBE 18B at 2026-01-14 20:29:48.015687,
 Event Uplink, station KSAT Troll to sat AEROCUBE 18B at 2026-01-14 20:29:48.015687,
 Event Uplink, station KSAT Troll to sat AEROCUBE 18B at 2026-01-14 20:29:48.015687,
 Event Uplink, station KSAT Troll to sat AEROCUBE 18B at 2026-01-14 20:29:48.015687,
 Event Uplink, station KSAT Troll to sat AEROCUBE 18B at 2026-01-14 20:29:48.015687,
 Event Uplink, station KSAT Troll to sat AEROCUBE 18B at 2026-01-14 20:29:48.015687,
 Event Uplink, station KSAT Troll to sat AEROCUBE 18B at 2026-01-14 20:29:48.015687,
 Event Uplink, station KSAT Troll to sat AEROCUBE 18B at 2026-01-14 20:29:48.015687,
 Event Uplink, station KSAT Troll to sat AEROCUBE 18B at 2026-01-

In [ ]:
retcode = 1
while (retcode !=0):
    retcode = world_with_schedulers.tick()
    # retcode = world_with_schedulers.tick(print_forbidden_prefixes=["Downlink", "End of downlink", "Unlock uplink", "Unlock satellite after obs"])

Executing Event Uplink, station KSAT Troll to sat AEROCUBE 18B at 2026-01-14 20:29:48.015687
Executing Event Uplink, station KSAT Troll to sat AEROCUBE 18B at 2026-01-14 20:29:48.015687
Executing Event Uplink, station KSAT Troll to sat AEROCUBE 18B at 2026-01-14 20:29:48.015687
Executing Event Uplink, station KSAT Troll to sat AEROCUBE 18B at 2026-01-14 20:29:48.015687
Executing Event Uplink, station KSAT Troll to sat AEROCUBE 18B at 2026-01-14 20:29:48.015687
Executing Event Uplink, station KSAT Troll to sat AEROCUBE 18B at 2026-01-14 20:29:48.015687
Executing Event Uplink, station KSAT Troll to sat AEROCUBE 18B at 2026-01-14 20:29:48.015687
Executing Event Uplink, station KSAT Troll to sat AEROCUBE 18B at 2026-01-14 20:29:48.015687
Executing Event Uplink, station KSAT Troll to sat AEROCUBE 18B at 2026-01-14 20:29:48.015687
Executing Event Uplink, station KSAT Troll to sat AEROCUBE 18B at 2026-01-14 20:29:48.015687
Executing Event Uplink, station KSAT Troll to sat AEROCUBE 18B at 2026

In [ ]:
def request_statistics(requests_pd):
    total_requests_no = len(requests_pd)
    all_statuses = set(requests_pd.status.values)    
    for s in all_statuses:
        # matching_statuses = sum([1 if (r['status']==s) else 0 for r in requests.values()])
        matching_statuses = len(requests_pd[requests_pd.status==s])
        print("{}/{} ({}%) of requests are in status {}".format(matching_statuses,total_requests_no, matching_statuses/total_requests_no*100, s))
    # X/Y requests have >1 successful observation
    unique_requests = set(requests_pd.request)
    unique_requests_no = len(unique_requests)
    fulfilled_unique_requests_no = 0
    for ur in unique_requests:
        matching_observation_statuses = requests_pd[(requests_pd['request']==ur) & (requests_pd['status']=="OK! Data received")]
        if len(matching_observation_statuses):
            fulfilled_unique_requests_no += 1
    print(" {}/{} ({}%) unique requests have at least one successful observation".format(fulfilled_unique_requests_no, unique_requests_no, fulfilled_unique_requests_no/unique_requests_no*100))

In [ ]:
print("LOFT")
request_statistics(scheduler_LOFT._requests)

print("Ubotica")
request_statistics(scheduler_ubotica._requests)

print("AC")
request_statistics(scheduler_aerospace._requests)




LOFT
1/50 (2.0%) of requests are in status ObservationStatus.ALL_OBSERVATION_OPPORTUNITIES_ARE_CONFLICTING
49/50 (98.0%) of requests are in status ObservationStatus.DATA_RECEIVED
 0/50 (0.0%) unique requests have at least one successful observation
Ubotica
9/50 (18.0%) of requests are in status ObservationStatus.ALL_OBSERVATION_OPPORTUNITIES_ARE_CONFLICTING
41/50 (82.0%) of requests are in status ObservationStatus.DATA_RECEIVED
 0/50 (0.0%) unique requests have at least one successful observation
AC
50/50 (100.0%) of requests are in status ObservationStatus.DATA_RECEIVED
 0/50 (0.0%) unique requests have at least one successful observation


In [ ]:
sim_start_time = dt.datetime.now(dt.timezone.utc).replace(tzinfo=None)

satellites_LOFT = [
    Satellite("LOFT YAM-6", Orbital("YAM-6", tle_file="TLEs.txt")),
    Satellite("LOFT YAM-10", Orbital("YAM-10", tle_file="TLEs.txt")),
]

satellites_ubotica = [
    Satellite("Ubotica CogniSat-6 HAMMER", Orbital("HAMMER", tle_file="TLEs.txt")),
    ]

satellites_aerospace = [
        Satellite("AEROCUBE 18A", Orbital("AEROCUBE 18A", tle_file="TLEs.txt")),
        Satellite("AEROCUBE 18B", Orbital("AEROCUBE 18B", tle_file="TLEs.txt")),   
]

satellites_multischedulers = satellites_LOFT+satellites_ubotica+satellites_aerospace

In [ ]:
world_with_brokers = World(satellites = satellites_multischedulers, phenomena=phenomena_cities)

N_BACKGROUND_SAMPLES_PER_CONSTELLATION = 50

scheduler_LOFT = ConstellationGroundScheduler(satellites=satellites_LOFT, ground_stations=ground_stations, world=world_with_brokers, name="LOFT")
scheduler_ubotica = ConstellationGroundScheduler(satellites=satellites_ubotica, ground_stations=ground_stations, world=world_with_brokers, name="UBOTICA")
scheduler_aerospace = ConstellationGroundScheduler(satellites=satellites_aerospace, ground_stations=ground_stations, world=world_with_brokers, name="AC")

world_with_brokers.add_constellation(scheduler_LOFT)
world_with_brokers.add_constellation(scheduler_ubotica)
world_with_brokers.add_constellation(scheduler_aerospace)

sampled_world_cities_LOFT = cities_of_the_world.sample(n=N_BACKGROUND_SAMPLES_PER_CONSTELLATION,weights='population',axis=0, random_state=0)
sampled_world_cities_ubotica = cities_of_the_world.sample(n=N_BACKGROUND_SAMPLES_PER_CONSTELLATION,weights='population',axis=0, random_state=1)
sampled_world_cities_aerospace = cities_of_the_world.sample(n=N_BACKGROUND_SAMPLES_PER_CONSTELLATION,weights='population',axis=0, random_state=2)
obs_request_LOFT = [
    ObservationRequest(city[1].lng, city[1].lat, min_time=min_time, max_time=max_time, alt_km=0.307, request_name=city[1].city,)
    for city in sampled_world_cities_LOFT.iterrows()
]
obs_request_ubotica = [
    ObservationRequest(city[1].lng, city[1].lat, min_time=min_time, max_time=max_time, alt_km=0.307, request_name=city[1].city,)
    for city in sampled_world_cities_ubotica.iterrows()
]
obs_request_aerospace = [
    ObservationRequest(city[1].lng, city[1].lat, min_time=min_time, max_time=max_time, alt_km=0.307, request_name=city[1].city,)
    for city in sampled_world_cities_aerospace.iterrows()
]


for request in obs_request_LOFT:
    scheduler_LOFT.schedule_request(
        request=request,
        current_time=sim_start_time,
        callback_request_scheduled=lambda obs: print("[LOFT] Request {} scheduled for observation {}!".format(request, obs)),
        callback_request_ready=lambda dp: print("[LOFT] Request {} ready with DP {}!".format(request, dp)),
        )
    
for request in obs_request_ubotica:
    scheduler_ubotica.schedule_request(
        request=request,
        current_time=sim_start_time,
        callback_request_scheduled=lambda obs: print("[UBOTICA] Request {} scheduled for observation {}!".format(request, obs)),
        callback_request_ready=lambda dp: print("[UBOTICA] Request {} ready with DP {}!".format(request, dp)),
        )
    
for request in obs_request_aerospace:
    scheduler_aerospace.schedule_request(
        request=request,
        current_time=sim_start_time,
        callback_request_scheduled=lambda obs: print("[AC] Request {} scheduled for observation {}!".format(request, obs)),
        callback_request_ready=lambda dp: print("[AC] Request {} ready with DP {}!".format(request, dp)),
        )
    
# scheduler_LOFT.schedule_downlinks(current_time=sim_start_time, max_time=sim_start_time+dt.timedelta(hours=36))
# scheduler_ubotica.schedule_downlinks(current_time=sim_start_time, max_time=sim_start_time+dt.timedelta(hours=36))
# scheduler_aerospace.schedule_downlinks(current_time=sim_start_time, max_time=sim_start_time+dt.timedelta(hours=36))

[LOFT] Scheduling request Request Korla
   No contacts for this satellite! Maybe we were too greedy
   No contacts for this satellite! Maybe we were too greedy
   No contacts for this satellite! Maybe we were too greedy
   No contacts for this satellite! Maybe we were too greedy
   No contacts for this satellite! Maybe we were too greedy
   No contacts for this satellite! Maybe we were too greedy
   No contacts for this satellite! Maybe we were too greedy
   No contacts for this satellite! Maybe we were too greedy
   No contacts for this satellite! Maybe we were too greedy
   No contacts for this satellite! Maybe we were too greedy
 [LOFT] Best request: None with None
   All observation opportunities are conflicting
[LOFT] Scheduling request Request Đà Lạt
   No contacts for this satellite! Maybe we were too greedy
   No contacts for this satellite! Maybe we were too greedy
   No contacts for this satellite! Maybe we were too greedy
   No contacts for this satellite! Maybe we were too 

In [ ]:
broker = Broker(constellations=[scheduler_LOFT, scheduler_ubotica, scheduler_aerospace], world=world_with_brokers)

world_with_brokers.add_broker(broker)

In [ ]:
sampled_world_cities_broker = cities_of_the_world.sample(n=50,weights='population',axis=0, random_state=3)

number_of_submissions = 5

obs_requests_broker = [
    ObservationRequest(city[1].lng, city[1].lat, min_time=min_time, max_time=max_time, alt_km=0.307, request_name=city[1].city,)
    for city in sampled_world_cities_broker.iterrows()
]

for _req in obs_requests_broker:
    broker.schedule_request(_req, current_time=sim_start_time, number_of_submissions=number_of_submissions)

[Broker] scheduling request Request Kaduna
[UBOTICA] Scheduling request Request Kaduna
   No contacts for this satellite! Maybe we were too greedy
 [UBOTICA] Best request: None with None
   All observation opportunities are conflicting
 [Broker] received UNscheduling of request Request Kaduna, pass Pass start: 2026-01-15 01:56:57.003537, highest: 2026-01-15 02:00:55.003081, fall: 2026-01-15 02:04:58.158852, from UBOTICA
[AC] Scheduling request Request Kaduna
   No contacts for this satellite! Maybe we were too greedy
 [AC] Best request: None with None
   All observation opportunities are conflicting
 [Broker] received UNscheduling of request Request Kaduna, pass Pass start: 2026-01-14 21:36:43.091529, highest: 2026-01-14 21:42:24.318859, fall: 2026-01-14 21:47:58.527617, from AC
[LOFT] Scheduling request Request Kaduna
   No contacts for this satellite! Maybe we were too greedy
 [LOFT] Best request: None with None
   All observation opportunities are conflicting
 [Broker] received UNsc

In [ ]:
retcode = 1
while (retcode !=0):
    # print("Tick!")
    retcode = world_with_brokers.tick(print_forbidden_prefixes=["Downlink", "End of downlink", "Unlock uplink", "Unlock satellite after obs"])
    # print(world_with_scheduler.events)

No more events


In [ ]:
request_statistics(broker._requests)

1/176 (0.5681818181818182%) of requests are in status ObservationStatus.NO_OBSERVATION_OPPORTUNITIES
175/176 (99.43181818181817%) of requests are in status ObservationStatus.ALL_OBSERVATION_OPPORTUNITIES_ARE_CONFLICTING
 0/50 (0.0%) unique requests have at least one successful observation


In [ ]:
retell_history(world_with_brokers)

In [ ]:
def plot_event(_chronicle: dict, world: World, ax=None):
    if ax is None:
        figglobal = plt.figure(figsize=(10,5))
        ax = figglobal.add_subplot(1,1,1, projection=ccrs.Robinson())
        ax.set_global()
        ax.coastlines()

    _time_to_plot_ground_track = dt.timedelta(seconds=15*60)
    _dt_to_plot_ground_track = dt.timedelta(seconds=60)
    time_steps_for_plotting = [_chronicle['time']- _dt_to_plot_ground_track*i for i in range(int(math.ceil(_time_to_plot_ground_track/_dt_to_plot_ground_track)))]

    ax.text(0,0,"{}".format(_chronicle['time']), transform=ax.transAxes)
    
    constellation_palette = cmap['viridis'].resampled(len(world.constellations))

    for phenomenon in _chronicle['phenomena']:
        if (phenomenon.start_time<_chronicle.time and phenomenon.end_time>_chronicle.time): 
            ax.plot(phenomenon.lon_deg, phenomenon.lat_deg, 'D', transform=ccrs.PlateCarree(), color='m')

    for constellation_ix, constellation in enumerate(world.constellations):
        constellation_color = constellation_palette(constellation_ix/len(world.constellations))

        # Plot the ground stations
        for ground_station in constellation.ground_stations:
            ax.plot(float(ground_station.lon_deg), float(ground_station.lat_deg), '*', transform=ccrs.PlateCarree(), color=constellation_color)

        # Plot the satellites
        for satellite in constellation.satellites:
            _orbit = satellite.orbit
            _llas = [_orbit.get_lonlatalt(t) for t in time_steps_for_plotting]
            # ax.plot([lla[0] for lla in _llas],[lla[1] for lla in _llas],transform=ccrs.Geodetic(), color=constellation_color)
            for lla_ix in range(len(time_steps_for_plotting)-1):
                ax.plot([_llas[lla_ix+1][0], _llas[lla_ix][0]], [_llas[lla_ix+1][1], _llas[lla_ix][1]], transform=ccrs.Geodetic(), color=constellation_color, alpha = (lla_ix+1)/len(_llas))

    if type(_chronicle['event'])==ObservationEvent:
        
        # Where are we looking - dashed line
        ax.plot(
            [_chronicle['event'].opportunity.lon_deg, _chronicle['event'].satellite.orbit.get_lonlatalt(_chronicle['time'])[0]],
            [_chronicle['event'].opportunity.lat_deg, _chronicle['event'].satellite.orbit.get_lonlatalt(_chronicle['time'])[1]],
            ':k',
            transform=ccrs.Geodetic()
        )

        # The ground footprint itself
        ax.plot(
            _chronicle['event'].opportunity.lon_deg,
            _chronicle['event'].opportunity.lat_deg,
            'Dr',
            markersize=10,
            transform=ccrs.Geodetic()
        )

        # The satellite location
        ax.plot(
            _chronicle['event'].satellite.orbit.get_lonlatalt(_chronicle['time'])[0],
            _chronicle['event'].satellite.orbit.get_lonlatalt(_chronicle['time'])[1],
            '.',
            markersize=10,
            transform=ccrs.Geodetic()
        )

        # The sensor footprint
        ground_footprint_llas = spacecraft_fov(_chronicle['time'], satellite, _chronicle['event'].opportunity.instrument, _chronicle['event'].opportunity)
        ground_footprint_poly = Polygon([(_lla[0], _lla[1]) for _lla in ground_footprint_llas])
        ax.add_patch(patch_from_polygon(ground_footprint_poly, fc=constellation_color, ec='none', alpha=0.5, transform=ccrs.Geodetic()))

    elif type(_chronicle['event'])==CommunicationEvent:
        # The line from the GS to the satellite
        ax.plot(
            [_chronicle['event'].station.lon_deg, _chronicle['event'].satellite.orbit.get_lonlatalt(_chronicle['time'])[0]],
            [_chronicle['event'].station.lat_deg, _chronicle['event'].satellite.orbit.get_lonlatalt(_chronicle['time'])[1]],
            '-.k',
            transform=ccrs.Geodetic()
        )
        # The station
        ax.plot(
            _chronicle['event'].station.lon_deg,
            _chronicle['event'].station.lat_deg,
            '*',
            markersize=10,
            transform=ccrs.Geodetic()
        )
        # And the satellite
        ax.plot(
            _chronicle['event'].satellite.orbit.get_lonlatalt(_chronicle['time'])[0],
            _chronicle['event'].satellite.orbit.get_lonlatalt(_chronicle['time'])[1],
            '.',
            markersize=10,
            transform=ccrs.Geodetic()
        )
    return ax


def plot_history(world: World):
    artists = []
    for _chronicle_ix, _chronicle in enumerate(world.history):
        _ax = plot_event(_chronicle, world)
        plt.savefig("History_{:05d}.png".format(_chronicle_ix))
        # artists.append(_ax)
        
    # plt.show()
        # if type(_chronicle['event'])==ObservationEvent:
        #     print("Observation: sat {} and opportunity {}".format(_chronicle['event'].satellite, _chronicle['event'].opportunity))
        # if type(_chronicle['event'])==CommunicationEvent:
        #     print("Communication: station {} to sat {} during pass {}".format(_chronicle['event'].station, _chronicle['event'].satellite, _chronicle['event'].comm_pass))

In [ ]:
# plot_history(world_with_brokers)
# # for i in range(10):
#     # plot_event(world_with_brokers.history[i], world_with_brokers)
